# The Image as Data — the CondadosAI unit, runnable

One notebook, one section per lesson, all on the same photograph: `_MDF0005.NEF`
from "Luxo Double Checker" in Mark Fairchild's HDR Photographic Survey.

Every Python snippet printed in the five posts has a cell here, so nothing has to be
retyped off the page. The C++ halves of the `CodeSwitch` blocks are page-only.

| Section | Lesson |
|---|---|
| 1 | [An image is an array](https://condados.ai/blog/an-image-is-an-array) |
| 2 | [Sampling the pixel grid](https://condados.ai/blog/sampling-the-pixel-grid) |
| 3 | [Quantization and bit depth](https://condados.ai/blog/quantization-bit-depth-and-banding) |
| 4 | [Colour](https://condados.ai/blog/image-formation-and-color-spaces) |
| 5 | [File formats and compression](https://condados.ai/blog/image-file-formats-and-compression) |

The survey's images are downloaded at run time and never redistributed. Its terms are
research and non-commercial publication, with the source acknowledged.


## Setup

Versions are pinned so this notebook keeps producing the numbers the articles publish.
Colab ships OpenCV 4.x by default; the pin below replaces it, and Colab may ask you to
restart the runtime afterwards — do that, then carry on from the next cell.


In [ ]:
!pip install -q "opencv-python>=5.0.0,<6.0.0" "numpy>=1.26" "matplotlib>=3.8" \
    "loguru>=0.7" "click>=8.1" "rawpy>=0.21" "exifread>=3.0" "xlrd>=2.0" \
    "openexr>=3.2" "pillow>=10.0"
!git clone -q https://github.com/CondadosAI/fundamentals-lab.git 2>/dev/null || true
%cd fundamentals-lab
!pip install -q -e .


In [ ]:
import cv2, numpy as np, rawpy, PIL
print("opencv", cv2.__version__, "| numpy", np.__version__,
      "| rawpy", rawpy.__version__, "| pillow", PIL.__version__)


Fetch the scene. This is unit 1.2's downloader — same scene, so no new fetcher.
The RAW exposures come out of a 14 GB archive one file at a time, so the first run
takes a few minutes.


In [ ]:
from click.testing import CliRunner
from fundamentals_lab.cli.sensing_download import sensing_download

result = CliRunner().invoke(sensing_download, [])
print(result.output[-800:] if result.output else result)


In [ ]:
from fundamentals_lab.imagedata import scene

frame = scene.load()          # develops the NEF at 8 and 16 bits
print(frame.mosaic.shape, frame.mosaic.dtype, "white level", frame.white_level)


## 1 — An image is an array

Three representations of one photograph, and the two conventions that bite:
`(row, column)` indexing and B, G, R channel order.


In [ ]:
img = frame.bgr8                  # what imread would hand you
print(img.shape, img.dtype)       # (2868, 4312, 3) uint8
h, w, c = img.shape

b, g, r = img[1910, 2474]         # row first, column second
print(int(b), int(g), int(r))     # 160 189 193


The `+60` arithmetic, three ways. The wrap is the one that turns highlights black.


In [ ]:
patch = img[1883:1971, 2448:2481]
brightest = patch.max()
print("brightest value ", brightest)                                    # 207
print("numpy wrap      ", np.uint8(brightest) + np.uint8(60))           # 11
print("opencv saturate ", cv2.add(patch, np.full_like(patch, 60)).max())  # 255
print("float then clip ", np.clip(patch.astype(np.float32) + 60, 0, 255).max())

# The obvious version is a trap: (patch + 60).max() is 255, not 11, because some
# pixel sat at exactly 195 and reached the ceiling without going over it.
print("the trap        ", (patch + np.uint8(60)).max())                 # 255


A slice is a view; fancy indexing is a copy.


In [ ]:
region = img[1883:1947, 2448:2512].copy()
before = region[0, 0].copy()
view = region[0:8, 0:8]
view[:] = 0                       # writes through to `region`
print(before, "->", region[0, 0])   # [167 194 199] -> [0 0 0]


In [ ]:
from fundamentals_lab.imagedata import arrays
import json
print(json.dumps(arrays.representations(frame)["rows"], indent=1)[:900])


## 2 — Sampling the pixel grid

`INTER_AREA` averages over the footprint; `INTER_NEAREST` just samples. The default,
`INTER_LINEAR`, reads a 2x2 neighbourhood — fine at x2, not at x8.


In [ ]:
small = cv2.resize(img, (w // 8, h // 8), interpolation=cv2.INTER_AREA)     # averages
naive = cv2.resize(img, (w // 8, h // 8), interpolation=cv2.INTER_NEAREST)  # samples
print("rms difference", np.sqrt(((small.astype(float) - naive) ** 2).mean()).round(3))


The measured ribbing, its drift down the table, and where each decimation puts it.


In [ ]:
from fundamentals_lab.imagedata import sampling

alias = sampling.aliasing(frame)
print("period", alias["base_period_px"], "px in rows", alias["window_rows_in_frame"])
for row in alias["factors"]:
    print(f"  x{row['factor']}: {row['samples_per_original_cycle']:.2f} samples/cycle,"
          f" apparent period {row['apparent_period_in_original_px']} px")


## 3 — Quantization and bit depth

Rounding, not truncation: truncating biases every value down by half a step.


In [ ]:
def requantize(values, bits, source_bits=12):
    """Round to `bits` levels, then put it back on the original scale."""
    step = 2 ** (source_bits - bits)
    return np.clip(np.rint(values / step) * step, 0, 2**source_bits - 1)

green = scene.plane(frame, "G1").astype(np.float64)
error = requantize(green, 8) - green
print("measured rms", np.sqrt((error ** 2).mean()).round(3), "| predicted", round(16 / np.sqrt(12), 3))


The crossing: where quantization error meets the noise unit 1.2 measured.


In [ ]:
from fundamentals_lab.imagedata import quantize

crossing = quantize.noise_floor_crossing()
for row in crossing["rows"]:
    print(f"  {row['channel']}: sigma {row['dark_sigma_dn']} DN -> {row['bits_at_crossing']} bits")
print("range", crossing["range_bits"], "| spare in a 12-bit file", crossing["spare_bits_in_a_12_bit_file"])


## 4 — Colour

Demosaicing, then the coordinate systems. `raw` here is a single-channel Bayer mosaic.


In [ ]:
raw = frame.mosaic
# The layout on this camera is RGGB; OpenCV names conversions by the opposite corner.
demosaiced = cv2.cvtColor((raw // 16).astype(np.uint8), cv2.COLOR_BayerBG2BGR)
print(demosaiced.shape, demosaiced.dtype)


In [ ]:
gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)   # (H, W), Rec. 601 luma
hsv   = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)    # H in [0,179], S,V in [0,255]
ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)  # channels: Y, Cr, Cb
lab   = cv2.cvtColor(img, cv2.COLOR_BGR2Lab)    # L in [0,255], a,b offset by 128
print(gray.shape, hsv.shape, ycrcb.shape, lab.shape)


In [ ]:
b, g, r = cv2.split(img)        # three single-channel images
merged  = cv2.merge([b, g, r])  # back to one 3-channel image
print(b.shape, merged.shape)


Patch 15 of the chart — the red one — through every space the lesson names.


In [ ]:
from fundamentals_lab.imagedata import colour

print(json.dumps(colour.worked_patch(frame), indent=1))


Segmenting by colour in HSV. Red wraps around hue 0, so two windows are unioned.


In [ ]:
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

mask1 = cv2.inRange(hsv, np.array([0, 120, 70]),   np.array([10, 255, 255]))
mask2 = cv2.inRange(hsv, np.array([170, 120, 70]), np.array([179, 255, 255]))
mask  = mask1 | mask2

result = cv2.bitwise_and(img, img, mask=mask)   # keep only the red pixels
print("red pixels:", int((mask > 0).sum()))


And the check against the colorimeter: three routes from photosites to Lab.


In [ ]:
accuracy = colour.accuracy(frame)
paths = accuracy["paths"]
print("naive sRGB        ", paths["naive_srgb"]["all"])
for name, variant in paths["published_matrix"]["variants"].items():
    print(f"  published {name:22s}", {k: variant[k] for k in ("mean", "median", "max")})
print("fitted, held out  ", paths["fitted_held_out"]["test"])


## 5 — File formats and compression

The sweep takes a minute or two: it encodes a 12-megapixel frame nine times.


In [ ]:
from fundamentals_lab.imagedata import formats

sweep = formats.sweep(frame)
for row in sweep["whole_frame"]["rows"]:
    if "error" in row:
        print(row["label"], row["error"]); continue
    print(f"  {row['label']:16s} {row['bytes']:>10,} B  edge F1 {row['edges']['f1']}")


The decoder comparison. Seven cases agree bit for bit; the 16-bit PNG does not,
because Pillow opens it in mode `RGB` and hands back 8-bit samples.


In [ ]:
sixteen_bit = frame.rgb16[1880:1980, 2440:2540][:, :, ::-1].copy()
cv2.imwrite("depth.png", sixteen_bit)

from PIL import Image
opencv = cv2.imread("depth.png", cv2.IMREAD_UNCHANGED)
pillow = np.array(Image.open("depth.png"))
print(opencv.dtype, opencv.max())              # uint16 53427
print(pillow.dtype, pillow.max())              # uint8   208   <- eight bits gone


In [ ]:
print(json.dumps(formats.decoder_disagreement(frame)["rows"], indent=1))


## Everything at once

This is the command that produces `output/image_data_numbers.json`, which every
number in all five posts comes from.


In [ ]:
from fundamentals_lab.cli.data_experiments import data_experiments

print(CliRunner().invoke(data_experiments, []).output[-1200:])
